In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
from IPython.display import display, Markdown

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(snakemake.input["util"]).parent))
sys.path.append(str(Path(snakemake.input["display_util"]).parent))

In [ ]:
from display_util import display_data_doc, display_long_data_doc, collist  # noqa: E402
from util import (  # noqa: E402
    drop_col_few_distinct,
    drop_duplicate_columns,
    common_translate,
    collapse_col,
)

In [ ]:
raw_followup = snakemake.input["before_filter_followup"]
after_filter_transplantation = snakemake.input["after_filter_transplantation"]
after_filter_recipients = snakemake.input["after_filter_recipients"]
targetpop = snakemake.input["targetpop"]
output_wide = snakemake.output["out_wide"]
output_long_et = snakemake.output["out_long_et"]
output_long_iqtig = snakemake.output["out_long_iqtig"]

In [ ]:
raw_followup_df = pd.read_parquet(raw_followup)
after_filter_transplantation_df = pd.read_parquet(after_filter_transplantation)
after_filter_recipients_df = pd.read_parquet(after_filter_recipients)
targetpop_df = pd.read_parquet(targetpop)

# Follow-Up Analysis and Data Preparation

In this notebook we prepare another dataset containing only follow-up
information for the recipients to do further analysis in other projects.

These are filtered to the target population described in the main report.

The columns `death_date` and `death_cause` come from the ET columns
`EBasisTodesdatumET` and `EBasisTodesursacheET` in the `empfaenger` table, 
which has been processed and filtered to the target population.

From the `transplantation` table, we keep the following columns:

 - `date_explanation_of_failure` (`TPostOPOrganversagenExplantationDateET`)
 - `failure_reason` (`TPostOPOrganversagenUrsacheET`)
 - `kidney_cause_of_death` (`TPostOPTodesursacheNIQTIG`)
 - `kidney_pancreas_bleeding` (`TBlutungenNPIQTIG`)
 - `kidney_pancreas_complications_general` (`TKomplikationIntraPostOperationAllgmeinNPIQTIG`)
 - `kidney_pancreas_complications_other` (`TKomplikationSonstigeNPIQTIG`)
 - `kidney_rejection` (`TPostOPAbstossungNIQTIG`)
 - `kidney_retransplant` (`TRetransplantationNIQTIG`)
 - `last_followup_date` (`TFollowUpLetztesDateET`)
 - `lost_to_followup` (`TFollowUpLostToFollowUpET`)
 - `post_operation_functional` (`TPostOPFunktionsaufnahmeTransplantatNIQTIG`)
 - `postop_organ_failure_date` (`TPostOPOrganversagenDateET`)


Here, we also use the processed version of the `transplantation` table,
which has been filtered to the target population. 

For the `followup_niere`
table, instead we used the original `followup_niere` with changed names
for this document and processing. The columns we keep are:

- `acute_rejections_count` (`FNAbstossungAkutAnzahlET`)
- `acute_rejection_date` (`FNAbstossungAkutDateET`)
- `acute_rejection` (`FNAbstossungAkutET`)
- `rejections_count` (`FNAbstossungAnzahlET`)
- `treated_chronic_rejections_count` (`FNAbstossungChronischBehandeltAnzahlET`)
- `treated_chronic_rejection` (`FNAbstossungChronischBehandeltET`)
- `chronic_rejection` (`FNAbstossungChronischET`)
- `rejection_et` (`FNAbstossungET`)
- `hospitilization_date` (`FNAufnahmeKrankenhausDateET`)
- `blind_trial` (`FNBlindversuchImmunsuppressivaET`)
- `bleeding` (`FNBlutungET`)
- `date_et` (`FNDateET`)
- `date_iqtig` (`FNDateIQTIG`)
- `duration_years` (`FNDauerWertIQTIG`)
- `last_dialysis_date` (`FNDialyseDateDialyseLetzteET`)
- `dialysis` (`FNDialyseET`)
- `dialysis_after_operation_count` (`FNDialysePostTransplantationAnzahlET`)
- `survey_type` (`FNErhebungsartIQTIG`)
- `followup_type` (`FNFollowUpArtET`)
- `graft_function` (`FNFunktionGraftET`)
- `graft_function_delayed` (`FNFunktionGraftVerzoegertET`)
- `graft_failure_date` (`FNGraftversagenDateIQTIG`)
- `graft_failure` (`FNGraftversagenIQTIG`)
- `graft_failure_reason` (`FNGraftversagenUrsacheIQTIG`)
- `recipient_et_id_et` (`FNIdEmpfaengerNrETET`)
- `recipient_et_iqtig` (`FNIdEmpfaengerNrETIQTIG`)
- `transplant_et_id` (`FNIdTransplantationsNrET`)
- `other_complications` (`FNKomplikationenAndereET`)
- `other_diseases` (`FNKrankheitenAndereET`)
- `other_transplant_diseases` (`FNKrankheitenTransplantationsbedingtAndereET`)
- `patient_died` (`FNPatientVerstorbenIQTIG`)
- `rejection_iqtig` (`FNRejektionIQTIG`)
- `death_date` (`FNTodesdatumIQTIG`)
- `death_reason` (`FNTodesursacheIQTIG`)

This table was also filtered to the target population here.

In [ ]:
raw_followup_df["recipient_id"] = collapse_col(
    raw_followup_df.loc[:, ["recipient_et_id_et", "recipient_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
olres = raw_followup_df["recipient_id"].drop_duplicates()
assert not raw_followup_df["recipient_id"].isna().any(), "Missing recipient info"
assert (
    raw_followup_df["date_et"] - raw_followup_df["date_iqtig"]
).dropna().abs().max() == 0, "Date disagreements"
assert (
    not targetpop_df["recipient_et_id_et"].duplicated().any()
), "Repeat recipients currently can not be matched due to IQTIG data"

raw_followup_df = (
    pd.merge(
        raw_followup_df,
        targetpop_df.loc[:, ["recipient_et_id_et", "transplant_et_id"]],
        left_on="recipient_id",
        right_on="recipient_et_id_et",
    )
    .drop(columns="recipient_et_id_et_y")
    .rename(columns={"recipient_et_id_et_x": "recipient_et_id_et"})
)
raw_followup_df = (
    raw_followup_df[
        (raw_followup_df["transplant_et_id_y"] == raw_followup_df["transplant_et_id_x"])
        | raw_followup_df["transplant_et_id_x"].isna()
    ]
    .drop(columns="transplant_et_id_y")
    .rename(columns={"transplant_et_id_x": "transplant_et_id"})
    .drop_duplicates()
)

display(
    Markdown(
        f"""The filter process reduces the number of recipients in the `followup_niere` data ({olres.size}) and target population ({targetpop_df["recipient_et_id_et"].nunique()})
            to {raw_followup_df["recipient_id"].nunique()} in the processed data.
        """
    )
)

del targetpop, olres

raw_followup_df = common_translate(
    raw_followup_df, snakemake.config["data"]["common_translations"]
)

To each column, we add the institution as a prefix, so that we can easily identify the source of the data in the final dataset.

In [ ]:
rec_df_cols = {
    "death_date": "et_death_date",
    "death_cause": "et_death_cause",
}

rec_df = after_filter_recipients_df.loc[:, list(rec_df_cols)].rename(
    columns=rec_df_cols
)

transpl_df_cols = {
    "date_explanation_of_failure": "et_date_explanation_of_failure",
    "failure_reason": "et_failure_reason",
    "kidney_cause_of_death": "iqtig_kidney_cause_of_death",
    "kidney_pancreas_bleeding": "iqtig_kidney_pancreas_bleeding",
    "kidney_pancreas_complications_general": "iqtig_kidney_pancreas_complications_general",
    "kidney_pancreas_complications_other": "iqtig_kidney_pancreas_complications_other",
    "kidney_rejection": "iqtig_kidney_rejection",
    "kidney_retransplant": "iqtig_kidney_retransplant",
    "last_followup_date": "et_last_followup_date",
    "lost_to_followup": "et_lost_to_followup",
    "post_operation_functional": "iqtig_post_operation_functional",
    "postop_organ_failure_date": "et_postop_organ_failure_date",
}

transpl_df = after_filter_transplantation_df.loc[:, list(transpl_df_cols)].rename(
    columns=transpl_df_cols
)

followup_df_cols = {
    "acute_rejections_count": "et_acute_rejections_count",
    "acute_rejection_date": "et_acute_rejection_date",
    "acute_rejection": "et_acute_rejection",
    "rejections_count": "et_rejections_count",
    "treated_chronic_rejections_count": "et_treated_chronic_rejections_count",
    "treated_chronic_rejection": "et_treated_chronic_rejection",
    "chronic_rejection": "et_chronic_rejection",
    "rejection_et": "et_rejection_et",
    "hospitilization_date": "et_hospitilization_date",
    "blind_trial": "et_blind_trial",
    "bleeding": "et_bleeding",
    "date_et": "et_date",
    "date_iqtig": "iqtig_date",
    "duration_years": "iqtig_duration_years",
    "last_dialysis_date": "et_last_dialysis_date",
    "dialysis": "et_dialysis",
    "dialysis_after_operation_count": "et_dialysis_after_operation_count",
    "survey_type": "iqtig_survey_type",
    "followup_type": "et_followup_type",
    "graft_function": "et_graft_function",
    "graft_function_delayed": "et_graft_function_delayed",
    "graft_failure_date": "iqtig_graft_failure_date",
    "graft_failure": "iqtig_graft_failure",
    "graft_failure_reason": "iqtig_graft_failure_reason",
    "recipient_et_id_et": "et_recipient_et_id",
    "recipient_et_iqtig": "iqtig_recipient_et_id",
    "transplant_et_id": "et_transplant_et_id",
    "other_complications": "et_other_complications",
    "other_diseases": "et_other_diseases",
    "other_transplant_diseases": "et_other_transplant_diseases",
    "patient_died": "iqtig_patient_died",
    "rejection_iqtig": "iqtig_rejection",
    "death_date": "iqtig_death_date",
    "death_reason": "iqtig_death_reason",
}

followup_df = raw_followup_df.loc[:, list(followup_df_cols)].rename(
    columns=followup_df_cols
)

del raw_followup_df, after_filter_transplantation_df, after_filter_recipients_df

In [ ]:
et_followup_df = followup_df.filter(regex="^et_").dropna(how="all")
display(
    Markdown(
        f"There were {et_followup_df.shape[0]} follow-up records with at least one ET field across {et_followup_df.shape[1]} ET columns."
    )
)
et_followup_df = drop_col_few_distinct(et_followup_df)
et_followup_df = drop_duplicate_columns(et_followup_df)
# sort columns by missigness (descending) and then alphabetically
et_followup_df = et_followup_df.reindex(
    sorted(et_followup_df.columns, key=lambda x: (et_followup_df[x].isna().sum(), x)),
    axis=1,
)
iqtig_followup_df = followup_df.filter(regex="^iqtig_").dropna(how="all")
display(
    Markdown(
        f"There were {iqtig_followup_df.shape[0]} follow-up records with at least one IQTIG field across {iqtig_followup_df.shape[1]} IQTIG columns."
    )
)
iqtig_followup_df = drop_col_few_distinct(iqtig_followup_df)
iqtig_followup_df = drop_duplicate_columns(iqtig_followup_df)

iqtig_followup_df = iqtig_followup_df.reindex(
    sorted(
        iqtig_followup_df.columns, key=lambda x: (iqtig_followup_df[x].isna().sum(), x)
    ),
    axis=1,
)

assert (et_followup_df.shape[1] + iqtig_followup_df.shape[1]) == followup_df.shape[
    1
], "Column count mismatch after splitting ET and IQTIG columns"
iqtig_followup_df.set_index(["iqtig_recipient_et_id"], inplace=True, drop=True)
et_followup_df.set_index(
    ["et_recipient_et_id", "et_transplant_et_id"], inplace=True, drop=True
)

With the longitudinal data in the `followup_niere` ET table, there are some columns, which actually contain
only one value per recipient and are therefore not really longitudinal.

In [ ]:
def non_na_unique_count(series):
    """Count the number of unique non-NA values in a pandas Series."""
    return series.dropna().nunique()


max_et = (
    et_followup_df.groupby("et_recipient_et_id").agg(non_na_unique_count).agg("max")
)
et_wide_cols = max_et.index[max_et == 1]
max_iqtig = (
    iqtig_followup_df.groupby("iqtig_recipient_et_id")
    .agg(non_na_unique_count)
    .agg("max")
)
iqtig_wide_cols = max_iqtig.index[max_iqtig == 1]
assert (
    iqtig_wide_cols.size == 0
), "There are IQTIG columns with more than one unique value per recipient, which is unexpected."

In [ ]:
def get_only_val_in_col(series):
    """Get the only unique non-NA value in a pandas Series"""
    unique_vals = series.dropna().unique()
    if unique_vals.size == 0:
        return None
    assert (
        unique_vals.size == 1
    ), f"Expected only one unique value in the series, but found {unique_vals.size}."
    return unique_vals[0]


et_wide_df = (
    et_followup_df.loc[:, et_wide_cols]
    .groupby("et_recipient_et_id")
    .agg(get_only_val_in_col)
)
et_followup_df.drop(columns=et_wide_cols, inplace=True)

bulletpoint_list = ""
for col in et_wide_cols:
    bulletpoint_list += f"- `{col}`\n"
display(
    Markdown(
        f"ET columns with only one unique value per recipient (moved to wide format):\n{bulletpoint_list}"
    )
)

The columns `iqtig_kidney_rejection` and `iqtig_kidney_retransplant` are
longitudinal columns in the `transplantation` table, they can have multiple
values per recipient.

In [ ]:
trans_long = (
    transpl_df.groupby("recipient_et_id_et").agg(non_na_unique_count).agg("max")
)
trans_long = trans_long[trans_long > 1]
transpl_df.loc[:, trans_long.index].dropna(how="all")

It appears like there is some mechanism in the IQTIG system, where a retransplant or rejection leads to another follow-up entry in the transplantation table. We drop these columns, as they were the only ones with multiple values per recipient in the `transplantation` table.

In [ ]:
transpl_df.drop(columns=trans_long.index, inplace=True)

In [ ]:
transpl_df_wide = transpl_df.groupby("recipient_et_id_et").agg(get_only_val_in_col)

We can now merge the wide ET and transplantation dataframes with the IQTIG dataframe to create the final wide format follow-up dataframe.

In [ ]:
wide_df = pd.merge(
    rec_df,
    et_wide_df,
    right_on="et_recipient_et_id",
    left_index=True,
    how="outer",
    validate="1:1",
)
wide_df = pd.merge(
    wide_df,
    transpl_df_wide,
    right_on="recipient_et_id_et",
    left_on="et_recipient_et_id",
    how="outer",
    validate="1:1",
)

date_df = targetpop_df.loc[:, ["recipient_et_id_et", "recipient_op_date"]]
wide_df = pd.merge(
    wide_df,
    date_df,
    right_on="recipient_et_id_et",
    left_on="et_recipient_et_id",
    how="left",
    validate="1:1",
)
date_cols = wide_df.filter(regex="date").columns
display(
    Markdown(
        f"Converting the following date columns to days since operation: {collist(date_cols)}\n"
    )
)
for col in date_cols:
    if col != "recipient_op_date":
        wide_df[col] = wide_df[col] - wide_df["recipient_op_date"]
wide_df.drop(columns=["recipient_op_date", "recipient_et_id_et"], inplace=True)
wide_df.rename({"et_recipient_et_id": "recipient_et_id"}, axis=1, inplace=True)

In [ ]:
display_data_doc(data=wide_df)

Next, we will store the longitudinal follow-up data from ET.

In [ ]:
et_followup_df = pd.merge(
    et_followup_df,
    date_df,
    right_on="recipient_et_id_et",
    left_on="et_recipient_et_id",
    how="left",
    validate="m:1",
)

date_cols = et_followup_df.filter(regex="date").columns
display(
    Markdown(
        f"Converting the following date columns to days since operation: {collist(date_cols)}\n"
    )
)
for col in date_cols:
    if col != "recipient_op_date":
        et_followup_df[col] = et_followup_df[col] - et_followup_df["recipient_op_date"]
et_followup_df.drop(columns=["recipient_op_date"], inplace=True)
et_followup_df.rename({"recipient_et_id_et": "recipient_et_id"}, axis=1, inplace=True)

In [ ]:
display_long_data_doc(
    data=et_followup_df,
    idcols=["recipient_et_id"],
    datecol="et_date",
    typecol="et_followup_type",
)

Next we store the longitudinal follow-up data from IQTIG.

In [ ]:
iqtig_followup_df = pd.merge(
    iqtig_followup_df,
    date_df,
    right_on="recipient_et_id_et",
    left_on="iqtig_recipient_et_id",
    how="left",
    validate="m:1",
)
date_cols = iqtig_followup_df.filter(regex="date").columns
display(
    Markdown(
        f"Converting the following date columns to days since operation: {collist(date_cols)}\n"
    )
)
for col in date_cols:
    if col != "recipient_op_date":
        iqtig_followup_df[col] = (
            iqtig_followup_df[col] - iqtig_followup_df["recipient_op_date"]
        )
iqtig_followup_df.drop(columns=["recipient_op_date"], inplace=True)
iqtig_followup_df.rename(
    {"recipient_et_id_et": "recipient_et_id"}, axis=1, inplace=True
)

In [ ]:
display_long_data_doc(
    data=iqtig_followup_df,
    idcols=["recipient_et_id"],
    datecol="iqtig_date",
    typecol="iqtig_duration_years",
)

We save  the three datasets to CSV files for further analysis in other projects.

In [ ]:
wide_df.to_csv(output_wide, index=False)
et_followup_df.to_csv(output_long_et, index=False)
iqtig_followup_df.to_csv(output_long_iqtig, index=False)